## Training notebook

In [1]:
import gym
import highway_env

# from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO

%load_ext tensorboard
import datetime, os
from ipywidgets import interact, widgets
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from typing import Callable

### Select operating system

In [2]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['UNIX', 'WINDOWS'])

interactive(children=(Dropdown(description='desired_os', options=('UNIX', 'WINDOWS'), value='UNIX'), Output())…

<function __main__.f(desired_os)>

In [3]:
OUTPUT_PATH = '/Users/fornerispighetti/HighwayDRL/training_output/' if os_id.value == "UNIX" else 'C:/Users/pigo/Desktop/HighwayDRL/training_output/'

### Select environment

In [ ]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['dm-env-v0','acc-dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0','multipleO-dm-env-v0'])

In [4]:

# Create and wrap the environment
env = gym.make(env_id.value)

# Save a checkpoint every 1000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=False, render=False)

In [5]:
def linear_schedule(initial_value: float) -> Callable[[float], float]:
    """
    Linear learning rate schedule.

    :param initial_value: Initial learning rate.
    :return: schedule that computes
      current learning rate depending on remaining progress
    """
    def func(progress_remaining: float) -> float:
        """
        Progress will decrease from 1 (beginning) to 0.

        :param progress_remaining:
        :return: current learning rate
        """
        return progress_remaining * initial_value

    return func

In [6]:
ilr = 5e-4
# PPO parameters
model = RecurrentPPO('MlpLstmPolicy', env, tensorboard_log=f'{OUTPUT_PATH}logdir', learning_rate=linear_schedule(ilr), verbose=1)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [7]:
num_timesteps = '2e4'

# Train the agent for n steps
model.learn(total_timesteps=int(num_timesteps), callback=[checkpoint_callback, eval_callback])

# Save the trained agent
model.save(f'{OUTPUT_PATH}models/RECppo_RL_{str(os_id.value)}_{num_timesteps}_{env_id.value}_{str(ilr)}_{datetime.datetime.today().day}-{datetime.datetime.today().month}')

Logging to C:/Users/pigo/Desktop/HighwayDRL/training_output/logdir\RecurrentPPO_11
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 28.8     |
|    ep_rew_mean     | 14.2     |
| time/              |          |
|    fps             | 24       |
|    iterations      | 1        |
|    time_elapsed    | 5        |
|    total_timesteps | 128      |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 22.5       |
|    ep_rew_mean          | 10.7       |
| time/                   |            |
|    fps                  | 25         |
|    iterations           | 2          |
|    time_elapsed         | 10         |
|    total_timesteps      | 256        |
| train/                  |            |
|    approx_kl            | 0.25897178 |
|    clip_fraction        | 0.55       |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.495     |
|  

c:\Users\pigo\miniconda3\envs\highway\lib\site-packages\stable_baselines3\common\evaluation.py:65: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=1000, episode_reward=63.68 +/- 34.82
Episode length: 88.20 +/- 42.15
----------------------------------------
| eval/                   |            |
|    mean_ep_length       | 88.2       |
|    mean_reward          | 63.7       |
| time/                   |            |
|    total_timesteps      | 1000       |
| train/                  |            |
|    approx_kl            | 0.20521155 |
|    clip_fraction        | 0.542      |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.434     |
|    explained_variance   | -0.000891  |
|    learning_rate        | 0.000764   |
|    loss                 | 22.6       |
|    n_updates            | 70         |
|    policy_gradient_loss | -0.384     |
|    value_loss           | 36.9       |
----------------------------------------
New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 47.4     |
|    ep_rew_mean     | 30       |
| time/         

### Continued learning

In [7]:
env_cl = gym.make("dm-env-v0")

In [8]:
model_cl = RecurrentPPO.load(f"{OUTPUT_PATH}models/RECppo_RL_WINDOWS_3e4_OVTKenv_cl2_15-6-2022", env=env_cl, tensorboard_log=f"{OUTPUT_PATH}logdir")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [9]:
# Save a checkpoint every 1000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env_cl, best_model_save_path=f'{OUTPUT_PATH}eval_logs',
                             log_path=f'{OUTPUT_PATH}eval_logs', eval_freq=1000,
                             deterministic=False, render=False)

In [10]:
model_cl.learn(total_timesteps=int(1e5), callback=[checkpoint_callback, eval_callback])

model_cl.save(f'{OUTPUT_PATH}models/RECppo_RL_{str(os_id.value)}_1e5_DMenv_cl3_{datetime.datetime.today().day}-{datetime.datetime.today().month}-{datetime.datetime.today().year}')

Logging to C:/Users/pigo/Desktop/HighwayDRL/training_output/logdir\RecurrentPPO_6
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 120      |
|    ep_rew_mean     | 17.1     |
| time/              |          |
|    fps             | 7        |
|    iterations      | 1        |
|    time_elapsed    | 18       |
|    total_timesteps | 128      |
---------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 120           |
|    ep_rew_mean          | 20.8          |
| time/                   |               |
|    fps                  | 6             |
|    iterations           | 2             |
|    time_elapsed         | 40            |
|    total_timesteps      | 256           |
| train/                  |               |
|    approx_kl            | 0.00090082607 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    e

c:\Users\pigo\miniconda3\envs\highway\lib\site-packages\stable_baselines3\common\evaluation.py:65: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=1000, episode_reward=48.22 +/- 11.61
Episode length: 100.20 +/- 25.71
------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 100          |
|    mean_reward          | 48.2         |
| time/                   |              |
|    total_timesteps      | 1000         |
| train/                  |              |
|    approx_kl            | 0.0004411321 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.0889      |
|    explained_variance   | -2.63e-05    |
|    learning_rate        | 0.000297     |
|    loss                 | 25.8         |
|    n_updates            | 3990         |
|    policy_gradient_loss | 0.168        |
|    value_loss           | 49.9         |
------------------------------------------
New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 111      |
|    ep_rew_m